# TC MSE Variance Budget Diagnostic Notebook

================================================================================ <br>
This notebook is a notebook implementation of the **TC_MSE** POD, converted from its
original driver-script form (`TC_MSE_Driver.py`, which called out to
`TC_snapshot_MSE_calc.py`, `Binning_and_compositing.py`, and `Plotting.py` via
`os.system`). Because the framework runs notebooks through `nbconvert`, the notebook
and `.py` implementations are functionally near-identical -- the main differences are
1) organizing the code into the standard section layout used by `example_notebook`
(Settings &rarr; Load &rarr; Compute &rarr; Plots), and 2) loading model data through the
intake-ESM data catalog instead of the older per-variable `<var>_var` environment
variables.

This file is part of the TC_MSE Diagnostic POD of the MDTF code package (see mdtf/MDTF-diagnostics/LICENSE.txt)

## TC MSE Variance Budget Analysis

Last Update: 9/4/2026 (notebook conversion)

This POD computes the column-integrated moist static energy (MSE) and the terms in the
budget for its spatial variance for tropical cyclones (TCs). The budget terms are
computed along the tracks of individual simulated TCs and then composited as a function
of TC intensity (maximum near-surface wind speed) for each snapshot. The results are
compared to equivalent calculations from 5 reanalysis datasets.

### Version & Contact info

- Version/revision information: version 2 (notebook conversion 9/2026); version 1 (2/22/2023)
- PI: Allison Wing, Florida State University, awing@fsu.edu
- Developer/point of contact (version 2): Allison Wing, Florida State University, awing@fsu.edu
- Developer/point of contact (version 1): Jarrett Starr, Florida State University, jstarr2@fsu.edu
- Other Contributors: Caitlin Dirkes, Suzana Camargo, Daehyun Kim

### Open source copyright agreement

The MDTF framework is distributed under the LGPLv3 license (see LICENSE.txt).

### Functionality

Pre-calculated TC track data is required as an input to the POD as obs data (a
formatted `.txt` file). The code extracts the TC center latitude/longitude, maximum
near-surface wind speed, and minimum sea level pressure at each time along each track,
then extracts the necessary variables to compute the column-integrated MSE and the
longwave, shortwave, and surface flux feedbacks in the budget for the spatial variance
of column-integrated MSE, in 10x10 degree boxes along the tracks of each TC. Snapshots
are trimmed to intensifying times (prior to each storm's lifetime maximum intensity)
equatorward of 30 degrees, binned by max wind speed in 3 m/s increments, and composited
over each bin. Model composites are compared to 5 reanalyses (ERA-5, ERA-Interim,
MERRA-2, CFSR, JRA-55) that have already been processed through this framework.

This notebook mirrors the 3 stages of the original driver:

1. **Section 3a** (was `TC_snapshot_MSE_calc.py`): extract track/model data and compute
   the MSE variance budget snapshots along the tracks of all TCs, saved per year.
2. **Section 3b** (was `Binning_and_compositing.py`): concatenate the yearly snapshot
   data, bin and composite by TC intensity, and box-average/normalize the feedbacks.
3. **Section 4** (was `Plotting.py`): generate the composite/azimuthal-mean/box-average/
   scatter plots using `Plotting_Functions.py`.

### Required programming language and libraries

* Python >= 3.10
* xarray, numpy, pandas, scipy, matplotlib, intake, yaml

### Required model output variables

3-D (time-lat-lon), 6-hourly: `rlds`, `rsds`, `rlus`, `rsus`, `hfls`, `hfss`, `rlut`, `rsut`, `rsdt`

4-D (time-plev-lat-lon), 6-hourly instantaneous: `ta` (K), `zg` (m), `hus` (1)

In addition to model output, this POD requires pre-computed TC track data and a
land-sea mask, both provided as observational/ancillary data (`OBS_DATA`), plus 5
reanalysis composite datasets used for comparison in the plots.

### References

1. Wing, A. A., Camargo, S. J., Sobel, A. H., Kim, D., Moon, Y., Murakami, H., Reed, K. A., Vecchi, G. A., Wehner, M. F., Zarzycki, C., & Zhao, M. (2019). Moist Static Energy Budget Analysis of Tropical Cyclone Intensification in High-Resolution Climate Models, Journal of Climate, 32(18), 6071-6095, https://doi.org/10.1175/JCLI-D-18-0599.1.
2. Dirkes, C.A., A.A. Wing, S.J. Camargo, and D. Kim (2023): Process-oriented diagnosis of tropical cyclones in reanalyses using a moist static energy variance budget, J. Climate, 36, 5293-5317, https://doi.org/10.1175/JCLI-D-22-0384.1.
3. Starr, J.C., A.A. Wing, S.J. Camargo, D. Kim, T.-Y. Lee, and J. Moon (2025): Using the Moist Static Energy Variance Budget to Evaluate Tropical Cyclones in Climate Models against Reanalyses and Satellite Observations, J. Climate, 38, 3353-3379, https://doi.org/10.1175/JCLI-D-24-0353.1.

================================================================================


# Section 1: Settings

In [1]:
# Import modules used in the POD
import os
import sys

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib
matplotlib.use('Agg')  # non-X windows backend
import matplotlib.pyplot as plt

import intake
import yaml

## Interactive vs. framework mode

The framework exports `case_env_file`, `POD_HOME`, `WORK_DIR`, `OBS_DATA` and the
per-case environment variables (`startdate`, `enddate`, `modelname`, `latres`,
`lonres`) before running this notebook. When developing/testing the notebook
standalone in Jupyter, `case_env_file` will not be set, so we fill in placeholder
values below. **Edit the paths in the `if interactive:` block for local testing** --
they are ignored when the notebook is run by the framework.

In [2]:
# True when running standalone in Jupyter for development/testing;
# False when launched by the framework via nbconvert.
interactive = "case_env_file" not in os.environ

if interactive:
    # ------------------------------------------------------------------
    # STANDALONE / INTERACTIVE SETTINGS -- edit these paths for local testing.
    # ------------------------------------------------------------------
    os.environ["POD_HOME"] = os.path.abspath(".")
    os.environ.setdefault("WORK_DIR", "/home/awing/mdtf/wkdir/TC_MSE")     # <- edit for local testing
    os.environ.setdefault("OBS_DATA", "/huracan/tank4/cornell/GCM/mdtf/inputdata/obs_data/TC_MSE")  # <- edit for local testing
    os.environ.setdefault("case_env_file","/home/awing/mdtf/MDTF-diagnostics/catalogs/GFDL.CM4.AMIP_catalog.json")
    os.environ.setdefault("startdate", "1979")
    os.environ.setdefault("enddate", "1983")
    os.environ.setdefault("modelname", "AM4")
    os.environ.setdefault("latres", "1")
    os.environ.setdefault("lonres", "1.25")

print(f"Running in {'interactive/standalone' if interactive else 'framework'} mode")

Running in interactive/standalone mode


**Note:** unlike the multi-case example PODs, `case_env_file` here does not need a
full `CASE_LIST` lookup for variable/coordinate names -- TC_MSE is a single-case POD
and the computation code below refers to the standard CMIP dimension names (`time`,
`lat`, `lon`, `plev`) directly, exactly as the original driver's scripts did.

In [3]:
# Receive the catalog location from the framework (or set directly above for
# interactive testing, since a single-case POD case_env_file may just point at the
# catalog file itself rather than a case_info.yml).
case_env_file = os.environ["case_env_file"]
assert os.path.isfile(case_env_file), f"case environment file not found: {case_env_file}"

if case_env_file.endswith(".json") or case_env_file.endswith(".csv"):
    # interactive shortcut: case_env_file was set directly to the catalog file
    cat_def_file = case_env_file
else:
    with open(case_env_file, 'r') as stream:
        try:
            case_info = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)
    cat_def_file = case_info['CATALOG_FILE']

In [4]:
# Analysis period and model grid settings (mirrors TC_MSE_Driver.py's use of
# os.environ["startdate"/"enddate"] and TC_snapshot_MSE_calc.py / Binning_and_compositing.py's
# use of os.environ["modelname"/"latres"/"lonres"])
start_year = int(os.environ["startdate"])
end_year = int(os.environ["enddate"])
modelname = str(os.environ["modelname"])
latres = float(os.environ["latres"])
lonres = float(os.environ["lonres"])

POD_HOME = os.environ["POD_HOME"]
WORK_DIR = os.environ["WORK_DIR"]
OBS_DATA = os.environ["OBS_DATA"]
os.makedirs(os.path.join(WORK_DIR, "model"), exist_ok=True)

# The 12 model variables TC_MSE needs, all at 6-hourly frequency (see settings.jsonc)
varlist = ["ta", "zg", "hus", "hfls", "hfss",
           "rlds", "rlus", "rlut", "rsds", "rsdt", "rsus", "rsut"]

print(f"model: {modelname}, years: {start_year}-{end_year}, grid: {latres} x {lonres} deg")

model: AM4, years: 1979-1983, grid: 1.0 x 1.25 deg


# Section 2: Load data

## What is in the data catalog?

In [6]:
# open the catalog using the file path provided by the framework/case environment
cat = intake.open_esm_datastore(cat_def_file)
cat

,unique
activity_id,1
assoc_files,0
institution_id,0
member_id,0
realm,1
variable_id,24
table_id,0
source_id,0
source_type,0
cell_methods,1


In [7]:
cat.df

,activity_id,assoc_files,institution_id,member_id,realm,variable_id,table_id,source_id,source_type,cell_methods,...,variant_label,grid_label,units,time_range,chunk_freq,standard_name,long_name,frequency,file_name,path
0,CMIP,NaN,NaN,NaN,atmos,hfls,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,surface_upward_latent_heat_flux,Surface Upward Latent Heat Flux,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
1,CMIP,NaN,NaN,NaN,atmos,hfss,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,surface_upward_sensible_heat_flux,Surface Upward Sensible Heat Flux,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
2,CMIP,NaN,NaN,NaN,atmos,hus,NaN,NaN,NaN,time: point,...,NaN,NaN,1.0,19790101:060000-19840101:000000,NaN,specific_humidity,Specific Humidity,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
3,CMIP,NaN,NaN,NaN,atmos,prw,NaN,NaN,NaN,time: point,...,NaN,NaN,kg m-2,19790101:060000-19840101:000000,NaN,atmosphere_water_vapor_content,Water Vapor Path,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
4,CMIP,NaN,NaN,NaN,atmos,ps,NaN,NaN,NaN,time: point,...,NaN,NaN,Pa,19790101:060000-19840101:000000,NaN,surface_air_pressure,Surface Air Pressure,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
5,CMIP,NaN,NaN,NaN,atmos,rlds,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,surface_downwelling_longwave_flux_in_air,Surface Downwelling Longwave Radiation,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
6,CMIP,NaN,NaN,NaN,atmos,rldscs,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,surface_downwelling_longwave_flux_in_air_assum...,Surface Downwelling Clear-Sky Longwave Radiation,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
7,CMIP,NaN,NaN,NaN,atmos,rlus,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,surface_upwelling_longwave_flux_in_air,Surface Upwelling Longwave Radiation,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
8,CMIP,NaN,NaN,NaN,atmos,rlut,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,toa_outgoing_longwave_flux,TOA Outgoing Longwave Radiation,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...
9,CMIP,NaN,NaN,NaN,atmos,rlutcs,NaN,NaN,NaN,time: point,...,NaN,NaN,W m-2,19790101:060000-19840101:000000,NaN,toa_outgoing_longwave_flux_assuming_clear_sky,TOA Outgoing Clear-Sky Longwave Radiation,6hr,NaN,/huracan/tank4/cornell/GCM/mdtf/inputdata/mode...


## Searching for the 6-hourly TC_MSE variables

The original driver's helper scripts opened each variable from a hardcoded,
per-variable environment variable (e.g. `os.environ["ta_var"]`) pointing directly at a
preprocessed file. Here we instead search the GFDL.CM4.AMIP catalog for each variable
at 6-hourly frequency and load it through `intake`/`xarray`, matching the approach used
by `example_notebook`/`example_multicase`.

In [8]:
var_datasets = {}
for v in varlist:
    subset = cat.search(variable_id=v, frequency="6hr")
    assert len(subset.df), f"No catalog entries found for variable '{v}' at 6hr frequency"
    ds_dict = subset.to_dataset_dict(
        progressbar=False,
        aggregate=False,
        xarray_open_kwargs={"decode_times": True, "use_cftime": True}
    )
    # single-case POD: exactly one dataset is expected per variable
    key = list(ds_dict)[0]
    var_datasets[v] = ds_dict[key]

var_datasets.keys()

dict_keys(['ta', 'zg', 'hus', 'hfls', 'hfss', 'rlds', 'rlus', 'rlut', 'rsds', 'rsdt', 'rsus', 'rsut'])

## Ancillary (non-catalog) observational inputs

TC track data, the fixed land-sea mask, and the 5 reanalysis composite datasets used
for comparison in the plots are not CMIP-style model diagnostic output, so (as in the
original driver) they continue to be read directly from `OBS_DATA` rather than through
the catalog.

In [9]:
# Track data: a formatted .txt file with TC center lat/lon, wind speed, and pressure
# along each track. Change ReadTrackData() in Section 3a if your track data format differs.
trackdata_file = os.path.join(OBS_DATA, "trackdata.txt")

# Land-sea mask: percent land, used to exclude (>20% land) grid points from composites
landsea_mask_file = os.path.join(OBS_DATA, "sftlf_fx_GFDL-CM4_amip_r1i1p1f1_gr1.nc")

assert os.path.isfile(trackdata_file), f"track data file not found: {trackdata_file}"
assert os.path.isfile(landsea_mask_file), f"land-sea mask file not found: {landsea_mask_file}"


# Section 3: Compute

## Step 3a: MSE budget snapshots along TC tracks

*(was `TC_snapshot_MSE_calc.py`)*

For each year, this loops over every tracked TC and every time along its track,
extracts a 10x10 degree box centered on the TC, and computes the column-integrated
moist static energy (MSE) and its budget feedback terms (longwave, shortwave, surface
enthalpy flux) as anomalies from the box average.

In [10]:
def boxavg(thing, lat, lon):
    # cosine-latitude weighted box average
    coslat_values = np.transpose(np.tile(np.cos(np.deg2rad(lat)), (len(lon), 1)))
    thing1 = thing * coslat_values
    thing2 = thing1 / thing1
    average = np.nansum(np.nansum(thing1, 0)) / np.nansum(np.nansum(coslat_values * thing2, 0))

    return average

In [11]:
def ReadTrackData(trackdata, start_year, end_year):
    df = pd.read_csv(trackdata, sep='\s+', header=None, names=
    ['lon', 'lat', 'windspeed (m/s)', 'pressure (hPa)', 'year', 'month', 'day', 'hour'])
    # Add flag so it knows where to start from
    df_starts = df[df['lon'] == 'start']
    # Start by assigning first storm in dataset with an ID of 1
    storm_id = 1
    for idx, num_steps in zip(df_starts.index, df_starts['lat'].values):
        # Add in column for storm ID
        df.loc[idx:idx + num_steps + 1, 'stormid'] = storm_id
        # Add 1 to storm ID each time you get to the end of a particular storm track to continue looping
        storm_id += 1

    # Drop the rows that have the starter variable
    df = df.dropna().reset_index(drop=True)  # only in the rows with start have NaN values, so this works

    # Adjust format of some columns
    df.loc[:, 'year'] = df.loc[:, 'year'].astype(int).astype(str)
    df.loc[:, 'month'] = df.loc[:, 'month'].astype(int).astype(str)
    df.loc[:, 'day'] = df.loc[:, 'day'].astype(int).astype(str)
    df.loc[:, 'hour'] = df.loc[:, 'hour'].astype(int).astype(str)

    # Adjust the times to match CMIP6 time read-in format
    df.loc[:, 'hour'] = np.where(df['hour'].astype(int) < 10, '0' + df['hour'], df['hour'])
    df.loc[:, 'day'] = np.where(df['day'].astype(int) < 10, '0' + df['day'], df['day'])
    df.loc[:, 'month'] = np.where(df['month'].astype(int) < 10, '0' + df['month'], df['month'])
    # Create a date stamp column in identical format to cftime conversion to string
    df.loc[:, 'Modeltime'] = df['year'] + '-' + df['month'] + '-' + df['day'] + ' ' + df['hour'] + ':00:00'

    # Find max storm ID number
    num_storms = int(max(df.iloc[:]['stormid']))

    # Creating list of storm IDs by year
    tracks_by_year = {year: [] for year in range(start_year, end_year + 1)}  # empty array of storm tracks by yr
    # Loop through all storms
    for storm in range(1, num_storms + 1):
        # Get list of characteristics of storm ID you're on
        ds_storm = df[df['stormid'] == storm]
        # Get years unique to that storm
        times = ds_storm['year'].values
        if (int(times[0]) < start_year and int(times[-1]) < start_year or
                int(times[0]) > end_year and int(times[-1]) > end_year):
            continue

        tracks_by_year[int(times[0])].append(storm)  # Append list of storms to start year

    return df, tracks_by_year

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1184036/1268050823.py:2: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(trackdata, sep='\s+', header=None, names=


In [12]:
# Read track data and group storm IDs by starting year
df, tracks_by_year = ReadTrackData(trackdata_file, start_year, end_year)
years = list(tracks_by_year)
years

/tmp/ipykernel_1184036/1268050823.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1979' '1979' '1979' ... '1983' '1983' '1983']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, 'year'] = df.loc[:, 'year'].astype(int).astype(str)
/tmp/ipykernel_1184036/1268050823.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1' '1' '1' ... '12' '12' '12']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, 'month'] = df.loc[:, 'month'].astype(int).astype(str)
/tmp/ipykernel_1184036/1268050823.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1' '1' '1' ... '11' '11' '11']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[:, '

[1979, 1980, 1981, 1982, 1983]

### Build coordinate index lookups and read the land-sea mask

In [16]:
# Use the ta dataset to build index lookups for time/lat/lon/plev -- all variables
# share the same model grid and time axis in this example.
ds = var_datasets["ta"]
tarray = ds.indexes['time'].to_datetimeindex()
tarray = tarray.astype(str)
itarray = pd.Index(tarray)
itlist = itarray.tolist()

# Now gather and put general lats/lons list into index format to use later for gathering
# lat/lon box data for a given time
lats = np.array(ds['lat'])
lons = np.array(ds['lon'])
plevs = np.array(ds['plev19'])
ilats = pd.Index(lats)
ilons = pd.Index(lons)
iplevs = pd.Index(plevs)
ilatlist = ilats.tolist()
ilonlist = ilons.tolist()
iplevlist = iplevs.tolist()

# From the track data gather the minimum MSLP for column-integrated MSE
minMSLP = min(df['pressure (hPa)']) * 100
minplev = ds['plev19'].sel(plev19=minMSLP, method='nearest')
upperlvlplev = min(ds['plev19'])
iminplev = iplevlist.index(minplev)
iupperplev = iplevlist.index(upperlvlplev)

# Gather the land-sea mask data (>20% land grid points are excluded from composites below)
mask_ds = xr.open_dataset(landsea_mask_file, decode_times=True, use_cftime=True)
lsm = mask_ds.sftlf
mask_ds.close()

# Variable datasets were already opened from the catalog in Section 2; reuse them
# directly here instead of re-opening each file with a separate xr.open_dataset() call
# (as the original driver's helper script did).
phi_ds = var_datasets["zg"]
T_ds = var_datasets["ta"]
q_ds = var_datasets["hus"]
hfls_ds = var_datasets["hfls"]
hfss_ds = var_datasets["hfss"]
rlds_ds = var_datasets["rlds"]
rlus_ds = var_datasets["rlus"]
rlut_ds = var_datasets["rlut"]
rsds_ds = var_datasets["rsds"]
rsdt_ds = var_datasets["rsdt"]
rsus_ds = var_datasets["rsus"]
rsut_ds = var_datasets["rsut"]

/tmp/ipykernel_1184036/3653341583.py:4: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  tarray = ds.indexes['time'].to_datetimeindex()


### Loop over years, storms, and track times to compute the budget snapshots

This is the computational core of the POD. For each year it allocates 4D/3D/2D save
arrays sized by (number of storms, max track length, lat box, lon box), then for every
storm and every time along its track: extracts the 10x10 degree box, computes the
column-integrated MSE (and its temperature/moisture contributions), the longwave and
shortwave column flux convergence, the surface moist enthalpy flux, and the box-average
anomalies and their variance/covariance products (the budget feedback terms). The
per-year results are kept in memory for Section 3b.

In [ ]:
reg_datasets_by_year = {}
budg_datasets_by_year = {}

for year in years:
    # Set up the 4 dimensions of the data arrays from model data
    # Latitude, Longitude amounts to get 10X10 deg box
    latlen = int(10 / latres + 1)  # Center lat position is one index, then 5 degrees up and 5 degrees down
    lonlen = int(10 / lonres + 1)  # Center lon position is one index, then 5 degrees left and 5 degrees right
    # Get the amount of storms in the year
    numstorms = len(tracks_by_year[year])
    # Get the maximum track observations across all storms in track data
    numsteps = max(df['stormid'].value_counts())

    # Create the 4-D arrays for all variables desired
    h_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    ClmnLWfluxConv_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    ClmnSWfluxConv_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    ClmnRadfluxConv_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    OLR_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistContrib_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempContrib_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hfls_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hfss_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    sfcMoistEnthalpyFlux_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hvar_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistvar_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempvar_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_LWanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_OLRanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_SWanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_RADanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_SEFanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_hflsanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hanom_hfssanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_LWanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_OLRanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_SWanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_RADanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_SEFanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_hflsanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hMoistanom_hfssanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_LWanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_OLRanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_SWanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_RADanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_SEFanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_hflsanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan
    hTempanom_hfssanom_save = np.ones((numstorms, numsteps, latlen, lonlen)) * np.nan

    # Non-Radiative variables (ex: slp, wind, years, months, days, hours, latbox, lonbox, clat, clon, etc.)
    # 3D variables
    latbox_save = np.ones((numstorms, numsteps, latlen)) * np.nan
    lonbox_save = np.ones((numstorms, numsteps, lonlen)) * np.nan

    # 2D variables
    maxwind_save = np.ones((numstorms, numsteps)) * np.nan
    minSLP_save = np.ones((numstorms, numsteps)) * np.nan
    Clat_save = np.ones((numstorms, numsteps)) * np.nan
    Clon_save = np.ones((numstorms, numsteps)) * np.nan
    year_save = np.ones((numstorms, numsteps)) * np.nan
    month_save = np.ones((numstorms, numsteps)) * np.nan
    day_save = np.ones((numstorms, numsteps)) * np.nan
    hour_save = np.ones((numstorms, numsteps)) * np.nan

    # Start Looping through the storms in the given year
    for s, storm in enumerate(tracks_by_year[year]):
        # Get the storm data for the storm the index is on
        stormdata = df[df['stormid'] == storm]
        # Get list/arrays of all storm track data for the specific storm
        times = stormdata['Modeltime'].values
        clats = np.array(stormdata.loc[:, 'lat'].astype(float))
        clons = np.array(stormdata.loc[:, 'lon'].astype(float))
        maxwind = np.array(stormdata.loc[:, 'windspeed (m/s)'])
        minSLP = np.array(stormdata.loc[:, 'pressure (hPa)'])
        yr = np.array(stormdata.loc[:, 'year'].astype(int))
        mo = np.array(stormdata.loc[:, 'month'].astype(int))
        d = np.array(stormdata.loc[:, 'day'].astype(int))
        hr = np.array(stormdata.loc[:, 'hour'].astype(int))
        # Start looping through all the times in the given storm we are on
        for t, time in enumerate(times):
            # Get time index from the model list of times that matches the track time currently on
            tind = itlist.index(times[t])
            # Get the clat/clon position that is closest to what is provided in track data
            clat = ds['lat'].sel(lat=clats[t], method='nearest')
            clon = ds['lon'].sel(lon=clons[t], method='nearest')
            # Get the index of the above found clat/clon
            iclat = ilatlist.index(clat)
            iclon = ilonlist.index(clon)
            # Now set up bounds of 10X10 deg box based on index spacing and must go 1 higher for largest bound
            latmax = iclat + int((latlen - 1) / 2 + 1)
            latmin = iclat - int((latlen - 1) / 2)
            lonmax = iclon + int((lonlen - 1) / 2 + 1)
            lonmin = iclon - int((lonlen - 1) / 2)
            # Now gather the lat/lon array for the box
            latbox = np.array(ds.lat.isel(lat=slice(latmin, latmax)))
            lonbox = np.array(ds.lon.isel(lon=slice(lonmin, lonmax)))
            # Now make a 2D array based on the land-sea mask that is zeros or NaN if >20%
            landsea_zerosNaNs = np.zeros((len(latbox), len(lonbox)))
            # Open the parent land-sea mask file from outside the loop that is sliced according to the
            # lat/lon bounds above
            landsea_sliced = np.squeeze(lsm.isel(lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            # Now loop through the sliced land-sea mask to assign NaNs to the grid points that are >20
            for i in range(0, len(latbox)):
                for j in range(0, len(lonbox)):
                    if (landsea_sliced[i][j] > 20):
                        landsea_zerosNaNs[i][j] = np.nan

            # Getting h data and calculating h
            g = 9.8  # m/s^2
            Cp = 1.00464e3  # J/(kg*K)
            Lv = 2.501e6  # J/kg
            # Getting geopotential
            phi = np.squeeze(phi_ds['zg'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            # Getting temp
            T = np.squeeze(T_ds['ta'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            # Getting q
            q = np.squeeze(q_ds['hus'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            # Calculate MSE
            mse = Cp * T + g * phi + Lv * q
            # Calculate MSE (Temperature Contribution)
            mseT = Cp * T
            # Calculate MSE (Moisture Contribution)
            mseMoist = Lv * q
            # Get dp and range of p from any of datasets above as they all use same p and indexing
            dp = -1 * np.diff(phi_ds['plev19'].isel(plev19=slice(iminplev - 1, iupperplev + 1)))  # To get a positive dp
            dptile = np.transpose(np.tile(dp, (mse.shape[1], mse.shape[2], 1)), (2, 0, 1))
            # Do column integration for column-integrated MSE
            h = sum(mse[iminplev:iupperplev + 1, :, :] * dptile) / g  # Column-Integrated MSE
            hTempContrib = sum(mseT[iminplev:iupperplev + 1, :,
                               :] * dptile) / g  # Column-Integrated MSE (Only Temperature Contribution)
            hMoistContrib = sum(mseMoist[iminplev:iupperplev + 1, :,
                                :] * dptile) / g  # Column-Integrated MSE (Only Moisture Contribution)

            # Net LW at sfc regular (rlus-rlds)
            rlus = np.squeeze(rlus_ds['rlus'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            rlds = np.squeeze(rlds_ds['rlds'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            netLWsfc = rlus - rlds

            # Net SW at sfc regular (rsds - rsus)
            rsds = np.squeeze(rsds_ds['rsds'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            rsus = np.squeeze(rsus_ds['rsus'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            netSWsfc = rsds - rsus

            # Net LW at TOA regular (rlut) (no downwelling of LW at TOA)
            rlut = np.squeeze(rlut_ds['rlut'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            netLWtoa = rlut

            # Net SW at TOA regular (rsdt - rsut) (only one downwelling SW at TOA variable, incident)
            rsdt = np.squeeze(rsdt_ds['rsdt'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            rsut = np.squeeze(rsut_ds['rsut'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            netSWtoa = rsdt - rsut

            # Column LW Flux Convergence regular (netLWsfc - netLWtoa)
            ClmnLWfluxConv = netLWsfc - netLWtoa

            # Column SW Flux Convergence regular (netSWtoa - netSWsfc)
            ClmnSWfluxConv = netSWtoa - netSWsfc

            # Column Radiative Flux Convergence regular (ClmnLWfluxConv + ClmnSWfluxConv)
            ClmnRadfluxConv = ClmnLWfluxConv + ClmnSWfluxConv

            # Surface Moist Enthalpy Flux, sfc upward latent heat flux + sfc upward sensible heat flux (hfls + hfss)
            hfls = np.squeeze(hfls_ds['hfls'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            hfss = np.squeeze(hfss_ds['hfss'].isel(time=tind, lat=slice(latmin, latmax), lon=slice(lonmin, lonmax)))
            SfcMoistEnthalpyFlux = hfls + hfss

            # Outgoing Longwave Radiation (OLR):
            OLR = rlut

            # MSE Budget Variables Calculations
            havg = boxavg(h, latbox, lonbox)
            hanom = h - havg

            hTempavg = boxavg(hTempContrib, latbox, lonbox)
            hTempanom = hTempContrib - hTempavg

            hMoistavg = boxavg(hMoistContrib, latbox, lonbox)
            hMoistanom = hMoistContrib - hMoistavg

            LWavg = boxavg(ClmnLWfluxConv, latbox, lonbox)
            LWanom = ClmnLWfluxConv - LWavg

            OLRavg = boxavg(OLR, latbox, lonbox)
            OLRanom = OLR - OLRavg

            SWavg = boxavg(ClmnSWfluxConv, latbox, lonbox)
            SWanom = ClmnSWfluxConv - SWavg

            RADavg = boxavg(ClmnRadfluxConv, latbox, lonbox)
            RADanom = ClmnRadfluxConv - RADavg

            SEFavg = boxavg(SfcMoistEnthalpyFlux, latbox, lonbox)
            SEFanom = SfcMoistEnthalpyFlux - SEFavg

            HFLSavg = boxavg(hfls, latbox, lonbox)
            HFLSanom = hfls - HFLSavg

            HFSSavg = boxavg(hfss, latbox, lonbox)
            HFSSanom = hfss - HFSSavg

            hvar = np.multiply(np.array(hanom), np.array(hanom))
            hMoistvar = np.multiply(np.array(hMoistanom), np.array(hMoistanom))
            hTempvar = np.multiply(np.array(hTempanom), np.array(hTempanom))

            hanomLWanom = np.multiply(np.array(hanom), np.array(LWanom))
            hanomOLRanom = np.multiply(np.array(hanom), np.array(OLRanom))
            hanomSWanom = np.multiply(np.array(hanom), np.array(SWanom))
            hanomRADanom = np.multiply(np.array(hanom), np.array(RADanom))
            hanomSEFanom = np.multiply(np.array(hanom), np.array(SEFanom))
            hanomHFLSanom = np.multiply(np.array(hanom), np.array(HFLSanom))
            hanomHFSSanom = np.multiply(np.array(hanom), np.array(HFSSanom))

            hMoistanomLWanom = np.multiply(np.array(hMoistanom), np.array(LWanom))
            hMoistanomOLRanom = np.multiply(np.array(hMoistanom), np.array(OLRanom))
            hMoistanomSWanom = np.multiply(np.array(hMoistanom), np.array(SWanom))
            hMoistanomRADanom = np.multiply(np.array(hMoistanom), np.array(RADanom))
            hMoistanomSEFanom = np.multiply(np.array(hMoistanom), np.array(SEFanom))
            hMoistanomHFLSanom = np.multiply(np.array(hMoistanom), np.array(HFLSanom))
            hMoistanomHFSSanom = np.multiply(np.array(hMoistanom), np.array(HFSSanom))

            hTempanomLWanom = np.multiply(np.array(hTempanom), np.array(LWanom))
            hTempanomOLRanom = np.multiply(np.array(hTempanom), np.array(OLRanom))
            hTempanomSWanom = np.multiply(np.array(hTempanom), np.array(SWanom))
            hTempanomRADanom = np.multiply(np.array(hTempanom), np.array(RADanom))
            hTempanomSEFanom = np.multiply(np.array(hTempanom), np.array(SEFanom))
            hTempanomHFLSanom = np.multiply(np.array(hTempanom), np.array(HFLSanom))
            hTempanomHFSSanom = np.multiply(np.array(hTempanom), np.array(HFSSanom))

            # Now save the data variables to its corresponding save name created in outer loop and add the
            # land-sea mask to convert >20% land grids to NaN
            # 4D Variables
            h_save[s, t, 0:len(latbox), 0:len(lonbox)] = h + landsea_zerosNaNs
            hMoistContrib_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistContrib + landsea_zerosNaNs
            hTempContrib_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempContrib + landsea_zerosNaNs
            ClmnLWfluxConv_save[s, t, 0:len(latbox), 0:len(lonbox)] = ClmnLWfluxConv + landsea_zerosNaNs
            ClmnSWfluxConv_save[s, t, 0:len(latbox), 0:len(lonbox)] = ClmnSWfluxConv + landsea_zerosNaNs
            ClmnRadfluxConv_save[s, t, 0:len(latbox), 0:len(lonbox)] = ClmnRadfluxConv + landsea_zerosNaNs
            OLR_save[s, t, 0:len(latbox), 0:len(lonbox)] = OLR + landsea_zerosNaNs
            hfls_save[s, t, 0:len(latbox), 0:len(lonbox)] = hfls + landsea_zerosNaNs
            hfss_save[s, t, 0:len(latbox), 0:len(lonbox)] = hfss + landsea_zerosNaNs
            sfcMoistEnthalpyFlux_save[s, t, 0:len(latbox), 0:len(lonbox)] = SfcMoistEnthalpyFlux + landsea_zerosNaNs
            # MSE Budget Variables
            hanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanom + landsea_zerosNaNs
            hvar_save[s, t, 0:len(latbox), 0:len(lonbox)] = hvar + landsea_zerosNaNs
            hMoistvar_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistvar + landsea_zerosNaNs
            hTempvar_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempvar + landsea_zerosNaNs
            hanom_LWanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomLWanom + landsea_zerosNaNs
            hanom_OLRanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomOLRanom + landsea_zerosNaNs
            hanom_SWanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomSWanom + landsea_zerosNaNs
            hanom_RADanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomRADanom + landsea_zerosNaNs
            hanom_SEFanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomSEFanom + landsea_zerosNaNs
            hanom_hflsanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomHFLSanom + landsea_zerosNaNs
            hanom_hfssanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hanomHFSSanom + landsea_zerosNaNs
            hMoistanom_LWanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomLWanom + landsea_zerosNaNs
            hMoistanom_OLRanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomOLRanom + landsea_zerosNaNs
            hMoistanom_SWanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomSWanom + landsea_zerosNaNs
            hMoistanom_RADanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomRADanom + landsea_zerosNaNs
            hMoistanom_SEFanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomSEFanom + landsea_zerosNaNs
            hMoistanom_hflsanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomHFLSanom + landsea_zerosNaNs
            hMoistanom_hfssanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hMoistanomHFSSanom + landsea_zerosNaNs
            hTempanom_LWanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomLWanom + landsea_zerosNaNs
            hTempanom_OLRanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomOLRanom + landsea_zerosNaNs
            hTempanom_SWanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomSWanom + landsea_zerosNaNs
            hTempanom_RADanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomRADanom + landsea_zerosNaNs
            hTempanom_SEFanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomSEFanom + landsea_zerosNaNs
            hTempanom_hflsanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomHFLSanom + landsea_zerosNaNs
            hTempanom_hfssanom_save[s, t, 0:len(latbox), 0:len(lonbox)] = hTempanomHFSSanom + landsea_zerosNaNs

            # 3D Variables
            latbox_save[s, t, 0:len(latbox)] = latbox
            lonbox_save[s, t, 0:len(lonbox)] = lonbox

            # 2D Variables
            maxwind_save[s, t] = maxwind[t]
            minSLP_save[s, t] = minSLP[t]
            Clat_save[s, t] = clat
            Clon_save[s, t] = clon
            year_save[s, t] = yr[t]
            month_save[s, t] = mo[t]
            day_save[s, t] = d[t]
            hour_save[s, t] = hr[t]

    ##### Save the variables, regular variables for each year and budget variables for each year
    regvars_ds = xr.Dataset(
        data_vars=dict(
            h=(['numstorms', 'numsteps', 'latlen', 'lonlen'], h_save,
               {'units': 'J/m^2', 'long_name': 'Column-Integrated MSE', '_FillValue': -9999,
                'GridType': 'Lat/Lon Grid'}),
            hMoistContrib=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistContrib_save,
                           {'units': 'J/m^2', 'long_name': 'Column-Integrated MSE', '_FillValue': -9999,
                            'GridType': 'Lat/Lon Grid'}),
            hTempContrib=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempContrib_save,
                          {'units': 'J/m^2', 'long_name': 'Column-Integrated MSE', '_FillValue': -9999,
                           'GridType': 'Lat/Lon Grid'}),
            ClmnLWfluxConv=(['numstorms', 'numsteps', 'latlen', 'lonlen'], ClmnLWfluxConv_save,
                            {'units': 'W/m^2', 'long_name': 'Column LW Flux Convergence', '_FillValue': -9999,
                             'GridType': 'Lat/Lon Grid'}),
            ClmnSWfluxConv=(['numstorms', 'numsteps', 'latlen', 'lonlen'], ClmnSWfluxConv_save,
                            {'units': 'W/m^2', 'long_name': 'Column SW Flux Convergence', '_FillValue': -9999,
                             'GridType': 'Lat/Lon Grid'}),
            ClmnRadfluxConv=(['numstorms', 'numsteps', 'latlen', 'lonlen'], ClmnRadfluxConv_save,
                             {'units': 'W/m^2', 'long_name': 'Column Radiative Flux Convergence', '_FillValue': -9999,
                              'GridType': 'Lat/Lon Grid'}),
            OLR=(['numstorms', 'numsteps', 'latlen', 'lonlen'], OLR_save,
                 {'units': 'W/m^2', 'long_name': 'Outgoing LW Radiation', '_FillValue': -9999,
                  'GridType': 'Lat/Lon Grid'}),
            hfls=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hfls_save,
                  {'units': 'W/m^2', 'long_name': 'Surface Upward Latent Heat Flux', '_FillValue': -9999,
                   'GridType': 'Lat/Lon Grid'}),
            hfss=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hfss_save,
                  {'units': 'W/m^2', 'long_name': 'Surface Upward Sensible Heat Flux', '_FillValue': -9999,
                   'GridType': 'Lat/Lon Grid'}),
            SEF=(['numstorms', 'numsteps', 'latlen', 'lonlen'], sfcMoistEnthalpyFlux_save,
                 {'units': 'W/m^2', 'long_name': 'Surface Moist Enthalpy Flux', '_FillValue': -9999,
                  'GridType': 'Lat/Lon Grid'}),
            latitude=(['numstorms', 'numsteps', 'latlen'], latbox_save,
                      {'units': 'Degrees', 'long_name': 'Latitude', '_FillValue': -9999,
                       'GridType': '1.0 deg Latitude Spacing'}),
            longitude=(['numstorms', 'numsteps', 'lonlen'], lonbox_save,
                       {'units': 'Degrees', 'long_name': 'Longitude', '_FillValue': -9999,
                        'GridType': '1.25 deg Longitude Spacing'}),
            maxwind=(['numstorms', 'numsteps'], maxwind_save,
                     {'units': 'm/s', 'long_name': 'Maximum Wind Speed', '_FillValue': -9999,
                      'GridType': 'Lat/Lon Grid'}),
            minSLP=(['numstorms', 'numsteps'], minSLP_save,
                    {'units': 'hPa', 'long_name': 'Minimum Sea Level Pressure', '_FillValue': -9999,
                     'GridType': 'Lat/Lon Grid'}),
            centerLat=(['numstorms', 'numsteps'], Clat_save,
                       {'units': 'Degrees', 'long_name': 'TC Center Latitude Position', '_FillValue': -9999,
                        'GridType': '1.0 deg Latitude Spacing'}),
            centerLon=(['numstorms', 'numsteps'], Clon_save,
                       {'units': 'Degrees', 'long_name': 'TC Center Longitude Position', '_FillValue': -9999,
                        'GridType': '1.25 deg Longitude Spacing'}),
            year=(['numstorms', 'numsteps'], year_save, {'units': 'Year of given storm', 'long_name': 'year'}),
            month=(['numstorms', 'numsteps'], month_save, {'units': 'Month of given storm', 'long_name': 'month'}),
            day=(['numstorms', 'numsteps'], day_save, {'units': 'Day of given storm', 'long_name': 'day'}),
            hour=(['numstorms', 'numsteps'], hour_save, {'units': 'Hour of given storm', 'long_name': 'hour'})
        )
    )
    #regvars_ds.to_netcdf(os.path.join(WORK_DIR, 'model', f'Model_Regular_Variables_{year}.nc'))
    reg_datasets_by_year[year] = regvars_ds
    regvars_ds.close()

    budgvars_ds = xr.Dataset(
        data_vars=dict(
            hanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_save,
                   {'units': 'J/m^2', 'long_name': 'Column-Integrated MSE Anomaly', '_FillValue': -9999,
                    'GridType': 'Lat/Lon Grid'}),
            hvar=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hvar_save,
                  {'units': 'J^2*m^-4', 'long_name': 'Variance of Anomaly of Column-Integrated MSE',
                   '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hMoistvar=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistvar_save,
                       {'units': 'J^2*m^-4', 'long_name': 'Variance of Anomaly of Moist Contribution of h',
                        '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hTempvar=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempvar_save,
                      {'units': 'J^2*m^-4', 'long_name': 'Variance of Anomaly of Temp Contribution of h',
                       '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_LWanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_LWanom_save,
                          {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of LW Anomaly and h Anomaly',
                           '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_OLRanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_OLRanom_save,
                           {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of OLR Anomaly and h Anomaly',
                            '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_SWanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_SWanom_save,
                          {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of SW Anomaly and h Anomaly',
                           '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_RADanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_RADanom_save,
                           {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of RAD Anomaly and h Anomaly',
                            '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_SEFanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_SEFanom_save,
                           {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of SEF Anomaly and h Anomaly',
                            '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_hflsanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_hflsanom_save,
                            {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of HFLS Anomaly and h Anomaly',
                             '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hanom_hfssanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hanom_hfssanom_save,
                            {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of HFSS Anomaly and h Anomaly',
                             '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hMoistanom_LWanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_LWanom_save,
                               {'units': 'J^2*m^-4*s^-1',
                                'long_name': 'Product of LW Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                'GridType': 'Lat/Lon Grid'}),
            hTempanom_LWanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_LWanom_save,
                              {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of LW Anomaly and hTempContrib Anomaly',
                               '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hMoistanom_SWanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_SWanom_save,
                               {'units': 'J^2*m^-4*s^-1',
                                'long_name': 'Product of SW Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                'GridType': 'Lat/Lon Grid'}),
            hTempanom_SWanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_SWanom_save,
                              {'units': 'J^2*m^-4*s^-1', 'long_name': 'Product of SW Anomaly and hTempContrib Anomaly',
                               '_FillValue': -9999, 'GridType': 'Lat/Lon Grid'}),
            hMoistanom_OLRanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_OLRanom_save,
                                {'units': 'J^2*m^-4*s^-1',
                                 'long_name': 'Product of OLR Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                 'GridType': 'Lat/Lon Grid'}),
            hTempanom_OLRanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_OLRanom_save,
                               {'units': 'J^2*m^-4*s^-1',
                                'long_name': 'Product of OLR Anomaly and hTempContrib Anomaly', '_FillValue': -9999,
                                'GridType': 'Lat/Lon Grid'}),
            hMoistanom_RADanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_RADanom_save,
                                {'units': 'J^2*m^-4*s^-1',
                                 'long_name': 'Product of RAD Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                 'GridType': 'Lat/Lon Grid'}),
            hTempanom_RADanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_RADanom_save,
                               {'units': 'J^2*m^-4*s^-1',
                                'long_name': 'Product of RAD Anomaly and hTempContrib Anomaly', '_FillValue': -9999,
                                'GridType': 'Lat/Lon Grid'}),
            hMoistanom_SEFanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_SEFanom_save,
                                {'units': 'J^2*m^-4*s^-1',
                                 'long_name': 'Product of SEF Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                 'GridType': 'Lat/Lon Grid'}),
            hTempanom_SEFanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_SEFanom_save,
                               {'units': 'J^2*m^-4*s^-1',
                                'long_name': 'Product of SEF Anomaly and hTempContrib Anomaly', '_FillValue': -9999,
                                'GridType': 'Lat/Lon Grid'}),
            hMoistanom_hflsanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_hflsanom_save,
                                 {'units': 'J^2*m^-4*s^-1',
                                  'long_name': 'Product of HFLS Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                  'GridType': 'Lat/Lon Grid'}),
            hTempanom_hflsanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_hflsanom_save,
                                {'units': 'J^2*m^-4*s^-1',
                                 'long_name': 'Product of HFLS Anomaly and hTempContrib Anomaly', '_FillValue': -9999,
                                 'GridType': 'Lat/Lon Grid'}),
            hMoistanom_hfssanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hMoistanom_hfssanom_save,
                                 {'units': 'J^2*m^-4*s^-1',
                                  'long_name': 'Product of HFSS Anomaly and hMoistContrib Anomaly', '_FillValue': -9999,
                                  'GridType': 'Lat/Lon Grid'}),
            hTempanom_hfssanom=(['numstorms', 'numsteps', 'latlen', 'lonlen'], hTempanom_hfssanom_save,
                                {'units': 'J^2*m^-4*s^-1',
                                 'long_name': 'Product of HFSS Anomaly and hTempContrib Anomaly', '_FillValue': -9999,
                                 'GridType': 'Lat/Lon Grid'}),
            latitude=(['numstorms', 'numsteps', 'latlen'], latbox_save,
                      {'units': 'Degrees', 'long_name': 'Latitude', '_FillValue': -9999,
                       'GridType': '1.0 deg Latitude Spacing'}),
            longitude=(['numstorms', 'numsteps', 'lonlen'], lonbox_save,
                       {'units': 'Degrees', 'long_name': 'Longitude', '_FillValue': -9999,
                        'GridType': '1.25 deg Longitude Spacing'}),
            maxwind=(['numstorms', 'numsteps'], maxwind_save,
                     {'units': 'm/s', 'long_name': 'Maximum Wind Speed', '_FillValue': -9999,
                      'GridType': 'Lat/Lon Grid'}),
            minSLP=(['numstorms', 'numsteps'], minSLP_save,
                    {'units': 'hPa', 'long_name': 'Minimum Sea Level Pressure', '_FillValue': -9999,
                     'GridType': 'Lat/Lon Grid'}),
            centerLat=(['numstorms', 'numsteps'], Clat_save,
                       {'units': 'Degrees', 'long_name': 'TC Center Latitude Position', '_FillValue': -9999,
                        'GridType': '1.0 deg Latitude Spacing'}),
            centerLon=(['numstorms', 'numsteps'], Clon_save,
                       {'units': 'Degrees', 'long_name': 'TC Center Longitude Position', '_FillValue': -9999,
                        'GridType': '1.25 deg Longitude Spacing'}),
            year=(['numstorms', 'numsteps'], year_save, {'units': 'Year of given storm', 'long_name': 'year'}),
            month=(['numstorms', 'numsteps'], month_save, {'units': 'Month of given storm', 'long_name': 'month'}),
            day=(['numstorms', 'numsteps'], day_save, {'units': 'Day of given storm', 'long_name': 'day'}),
            hour=(['numstorms', 'numsteps'], hour_save, {'units': 'Hour of given storm', 'long_name': 'hour'})
        )
    )
    #budgvars_ds.to_netcdf(os.path.join(WORK_DIR, 'model', f'Model_Budget_Variables_{year}.nc'))
    budg_datasets_by_year[year] = budgvars_ds
    budgvars_ds.close()

print(f"Computed MSE budget snapshots for {len(years)} year(s): {years}")

/home/awing/.conda/envs/_MDTF_base/lib/python3.12/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/tmp/ipykernel_1184036/1393468892.py:6: RuntimeWarning: invalid value encountered in scalar divide
  average = np.nansum(np.nansum(thing1, 0)) / np.nansum(np.nansum(coslat_values * thing2, 0))
/home/awing/.conda/envs/_MDTF_base/lib/python3.12/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/tmp/ipykernel_1184036/1393468892.py:6: RuntimeWarning: invalid value encountered in scalar divide
  average = np.nansum(np.nansum(thing1, 0)) / np.nansum(np.nansum(coslat_values * thing2, 0))
/home/awing/.conda/envs/_MDTF_base/lib/python3.12/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/tmp/ipykernel_1184036/1393468892.py:6: Runt

## Step 3b: Binning and compositing by TC intensity

*(was `Binning_and_compositing.py`)*

Concatenates the per-year snapshot datasets from Step 3a, adds normalized/box-averaged
versions of each feedback term, tags each snapshot with its TC intensity (max wind
speed), trims snapshots to before lifetime maximum intensity (LMI) and equatorward of
30 degrees, then bins and composites everything in 3 m/s wind speed bins.

In [ ]:
# 10x10 degree box-relative lat/lon grid used for the composite fields below.
# NOTE: renamed from the original scripts' `lats`/`lons` to `box_lats`/`box_lons` to avoid
# overwriting the full model-grid `lats`/`lons` arrays defined in Step 3a -- in the
# original 3-script driver these lived in separate processes/namespaces so the name
# reuse was harmless, but it would silently clobber data in a single notebook namespace.
box_lats = np.arange(-5, 5 + latres, latres)
box_lons = np.arange(-5, 5 + lonres, lonres)

# Reuse the per-year datasets already computed in memory in Step 3a 
ds_all = []
for y in range(start_year, end_year + 1):
    ds_merge = xr.merge([reg_datasets_by_year[y], budg_datasets_by_year[y]])
    ds_all.append(ds_merge)

# Concatenate the year files together so all variables are combined across all storms
data = xr.concat(ds_all, dim='numstorms')

# Get a list of the data variables in data to trim the data after lifetime maximum intensity (LMI)
Model_vars = list(data.keys())

# Grab the vmax variable to get the LMI itself and point of LMI for trimming to account only for
# intensification period
maxwinds = data['maxwind']
winds_list = []

In [ ]:
# Loop through the variables to pick out the feedbacks and add a normalized version of that variable
for var in Model_vars:
    if var[0:5] == 'hanom' or var[0:10] == 'hMoistanom' or var[0:10] == 'hTempanom' or var[0:4] == 'hvar':
        normvar = np.array(data[var])
        boxavrawvar = np.array(data[var])
        boxavvar = np.ones((len(maxwinds), len(maxwinds[0]))) * np.nan
        boxavnormvar = np.ones((len(maxwinds), len(maxwinds[0]))) * np.nan
        for s in range(len(maxwinds)):
            for t in range(len(maxwinds[s])):
                hvar = np.array(data.hvar[s][t][:][:])
                boxavghvar = boxavg(hvar, np.array(data.latitude[s][t][:]), np.array(data.longitude[s][t][:]))
                normvar[s][t][:][:] = normvar[s][t][:][:] / boxavghvar
                boxavvar[s][t] = boxavg(boxavrawvar[s][t][:][:], np.array(data.latitude[s][t][:]),
                                        np.array(data.longitude[s][t][:]))
                boxavnormvar[s][t] = boxavg(np.array(normvar[s][t][:][:]), np.array(data.latitude[s][t][:]),
                                            np.array(data.longitude[s][t][:]))
        data['norm' + var] = (['numstorms', 'numsteps', 'latlen', 'lonlen'], np.array(normvar[:][:][:][:]))
        data['boxav_' + var] = (['numstorms', 'numsteps'], np.array(boxavvar[:][:]))
        data['boxav_norm_' + var] = (['numstorms', 'numsteps'], np.array(boxavnormvar[:][:]))

In [ ]:
# Loop through all model storms and find the LMI, then tag each storm with its LMI for binning later
for s in range(len(maxwinds)):
    windmax = float(max(maxwinds[s][:]))
    windmaxindex = np.squeeze(np.where(maxwinds[s] == windmax))
    # Check if there are more than one maximum wind speed
    if windmaxindex.size >= 2:
        windmaxindex = int(windmaxindex[0])
    else:
        windmaxindex = int(np.squeeze(np.where(maxwinds[s] == windmax)))
    # Loop and have all the indices after the timestep of LMI be NaN for all vars
    for var in Model_vars:
        data[var][s, windmaxindex + 1:len(maxwinds[s]) + 1] = np.nan

    vmax_indiv_list = []
    for t in range(0, len(maxwinds[s])):
        # First check and NaN all variables at timesteps where TC center is outside 30 N/S
        if data.centerLat[s][t] > 30 or data.centerLat[s][t] < -30:
            for var in Model_vars:
                if data[var].ndim == 2:
                    data[var][s][t] = np.nan
                elif data[var].ndim == 3:
                    data[var][s][t][:] = np.nan
                else:
                    data[var][s][t][:][:] = np.nan
        # Get max wind at specific step to tag the steps for binning snapshot
        vmax_sel = maxwinds[s, t].values
        vmax = xr.full_like(data.h[s, t], float(vmax_sel)).rename('vmax')
        vmax_indiv_list.append(vmax)
    vmax_indiv_array = xr.concat(vmax_indiv_list, dim='numsteps')
    # Create the vmax tag variable
    winds_list.append(vmax_indiv_array)

# Update Model data with the vmax tag created above
model_winds_array = xr.concat(winds_list, dim='numstorms')
model_updated = xr.merge([data, model_winds_array])
# Stretch the boxav variables to 1 dimension and make a new stretched windmax variable
newvars = list(model_updated.keys())
for var in newvars:
    if var[0:5] == 'boxav':
        (model_updated)['new_' + var] = (['newsteps'], np.squeeze(np.reshape(np.array(model_updated[var]),
                                                                            (len(data.numstorms) * len(data.numsteps)))))

model_updated['new_maxwind'] = (['newsteps'], np.squeeze(np.reshape(np.array(model_updated['maxwind']),
                                                                   (len(data.numstorms) * len(data.numsteps)))))

In [ ]:
# Bin snapshots according to max wind speed bins
bins = np.arange(0, 66, 3)
# Set a count array to gather the sample size for each bin and all bins
count_denom = len(data.latitude[0][0]) * len(data.longitude[0][0])
bins_count = np.zeros(len(bins))
vmax2 = model_updated.vmax.copy(deep=True)
onedvmax = model_updated.new_maxwind.copy(deep=True)
for b, bin in enumerate(bins):
    upperbin = bin + 3
    # Variable to get the number of samples for the current bin (divide by the resolution dims multiplied together)
    count = (len(np.where((model_updated.vmax >= bin) & (model_updated.vmax < upperbin))[0]) / count_denom)
    bins_count[b] = count
    vmax2 = (xr.where((model_updated.vmax >= bin) & (model_updated.vmax < upperbin), b, vmax2))
    onedvmax = (xr.where((model_updated.new_maxwind >= bin) & (model_updated.new_maxwind < upperbin), b, onedvmax))
bin_ds = xr.Dataset(data_vars=dict(bins=(['numstorms', 'numsteps', 'latlen', 'lonlen'], vmax2.values)))
onedbin_ds = xr.Dataset(data_vars=dict(newbins=(['newsteps'], onedvmax.values)))
ds = xr.merge([model_updated, bin_ds, onedbin_ds])
ds = ds.set_coords(['bins'])
ds = ds.set_coords(['newbins'])

In [ ]:
# Get the mean of each bin for one composite image
bins = np.arange(0, 22, 1)
binlabels = np.arange(1.5, 66, 3)
dsbins = ds['bins'].values
dsnewbins = ds['newbins'].values
binmeans = {}
binboxavstdevs = {}
for var_name, values in ds.items():
    dvar = ds[var_name].values
    if len(np.shape(dvar)) == 4:
        avg_bin_list = []
        for b, bin in enumerate(bins):
            avg_bin_list.append(np.nanmean(np.where(dsbins == bin, np.array(dvar), np.nan), axis=(0, 1)))
        binmeans[var_name] = (['bin', 'lat', 'lon'], avg_bin_list)
    if len(np.shape(dvar)) == 1:
        avg_bin_list = []
        stdev_bin_list = []
        for b, bin in enumerate(bins):
            avg_bin_list.append(np.nanmean(np.where(dsnewbins == bin, np.array(var), np.nan), axis=(0)))
            stdev_bin_list.append(np.nanstd(np.where(dsnewbins == bin, np.array(dvar), np.nan), axis=(0)))
        binmeans[var_name] = (['bin'], avg_bin_list)
        binboxavstdevs[var_name] = (['bin'], stdev_bin_list)

# Bin means
binmeans['bin'] = (['bin'], binlabels)
binmeans['lat'] = (['lat'], box_lats)
binmeans['lon'] = (['lon'], box_lons)
binmeans['bincounts'] = ('bin', bins_count)
# Add relevant raw variables that have not been binned or composited
binmeans['maxwind'] = (['numstorms', 'numsteps'], np.array(data['maxwind']))
binmeans['minSLP'] = (['numstorms', 'numsteps'], np.array(data['minSLP']))

# Binned boxav stdevs
binboxavstdevs['bin'] = (['bin'], binlabels)
binboxavstdevs['bincounts'] = ('bin', bins_count)

modelbinavgdata = xr.Dataset(data_vars=binmeans, attrs={'description': 'Mean Binned Data'})
modelbinboxavstdevdata = xr.Dataset(data_vars=binboxavstdevs,
                                    attrs={'description': 'Standard Deviations of Binned Box Averaged Variables'})

modelbinavgdata.to_netcdf(os.path.join(WORK_DIR, 'model', 'Model_Binned_Composites.nc'))
modelbinboxavstdevdata.to_netcdf(os.path.join(WORK_DIR, 'model', 'Model_Binned_STDEVS_of_BoxAvgs.nc'))

print("Binning and compositing complete.")

# Section 4: Plots

*(was `Plotting.py`)*

`Plotting_Functions.py` contains ~1100 lines of matplotlib code for the spatial
composite panels, azimuthal-mean plots, box-averaged line plots, and scatter plots. As
with the "external script" technique demonstrated in Part 4 of the `example_notebook`
POD (used there to avoid notebook bloat), we keep it as an unmodified external module
in `POD_HOME` and import it rather than inlining ~1100 lines here. It reads the
`Model_Binned_Composites.nc` / `Model_Binned_STDEVS_of_BoxAvgs.nc` files written by
Step 3b above, plus the 5 reanalysis comparison datasets from `OBS_DATA`, at import
time.

In [ ]:
# the use of external python scripts can help prevent bloat in the notebook
sys.path.append(POD_HOME)
import Plotting_Functions as plot

In [ ]:
# This will plot the spatial composite panels (4 plot files saved)
plot.SpatialCompositePanels()

In [ ]:
# This will plot the azimuthal mean line plots (1 plot file saved)
plot.AzmeanPlotting()

In [ ]:
# This will plot the non-normalized and normalized box-averaged feedbacks
# as a function of bin. (2 plot files saved)
plot.BoxAvLinePlotting()

In [ ]:
# This will plot the scattering of non-normalized and normalized
# box-averaged feedbacks and percent of storms intensifying from one
# bin to the next. (1 plot file saved)
plot.BoxAvScatter()

# Wrap-up

In [ ]:
# Close the catalog and release large in-memory objects for garbage collection
cat.close()
var_datasets = None
reg_datasets_by_year = None
budg_datasets_by_year = None

print("Last log message by TC_MSE POD: finished successfully!")